# Create NIJ Awards (National Institute of Justice)

Creates National Institute of Justice awards from NIJ's own "List of Funded
Projects" at https://nij.ojp.gov/funding/awards/list (all awards since FY2003,
plus some earlier ones still open; method 5, static HTML list pages: one full
pass ordered by title plus one pass per Funding Category filter, see
`scripts/local/nij_to_s3.py`). The list has one row per award ACTION (original
award + each supplement, " - S" titles); the script folds 7,488 actions into
**6,596 award numbers**, of which **3,736 research / evaluation / fellowship
awards ship** (FY1993-2024, USD 2.87B, 100% title / recipient / amount / fiscal
year). No abstract or named PI is published (org-level recipient only).

**Scope filter (oxjob #1451, 2026-10-01 rule; applied in the script, rows kept
in the parquet with `excluded = '1'` and `exclusion_reason`):** exclude ONLY
clearly operational formula and backlog-reduction programmes:
- `formula` (2,226): Funding Category "Formula" (DNA Backlog Reduction / DNA
  Capacity Enhancement and Backlog Reduction, Coverdell formula, etc.)
- `coverdell_forensic_capacity` (607): the competitive half of the Paul Coverdell
  Forensic Science Improvement Grants, the same forensic-lab capacity programme
  as its formula half (equipment, overtime, accreditation, backlog elimination)
- `backlog_reduction` (14): DNA / casework backlog-reduction awards outside the
  Formula category (e.g. "No Suspect Casework DNA Backlog Reduction")
- `not_awarded` (13): status "Declined" / "No Award"
A solicitation naming research / evaluation / development / fellowship is never
excluded by the Coverdell or backlog patterns. Kept but operational-flavoured
competitive programmes (Solving Cold Cases with DNA, Postconviction DNA Testing,
Using DNA Technology to Identify the Missing) are flagged in the batch report.

**Amount / dates:** amount = sum over the award's actions (original + supplements;
`amount_original` kept in raw), USD. `start_year` = fiscal year of the original
action, `start_date` = Jan 1 of that year (§2.3 year-only pattern; NIJ makes
awards late in the fiscal year, so the calendar year matches). No end date.

**`funder_award_id` (§2.1.1):** the NIJ award number (`2019-DU-BX-0001`,
`15PNIJ-24-GG-01571-RESS`, interagency `DJO-NIJ-19-RO-0007`), the reference
grantees cite: 2,055 of 4,006 existing citation stubs for F4320337430 match a shipped
award number exactly (case-insensitive); only 4 stubs point at excluded (formula
DNA backlog) awards.

**Overlap check:** no USAspending ingest covers DOJ / OJP / NIJ (grep of
notebooks/awards and scripts/local); no existing direct ingest uses F4320337430; no
twin funder row (ROR 00v8p7w89 only on this F-id; Office of Justice Programs
F4320332508 and DOJ F4320306098 are parent bodies, not twins).

**Prerequisites:** run `scripts/local/nij_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/nij/nij_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320337430`
- display_name: National Institute of Justice
- ror_id / doi: from `openalex.funders.funders` (ror 00v8p7w89, doi 10.13039/100005289)

**Priority:** `537` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nij_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nij/nij_projects.parquet`;

In [ ]:
%sql
SELECT excluded, exclusion_reason, COUNT(*) as n, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as usd
FROM openalex.awards.nij_raw GROUP BY 1, 2 ORDER BY 1, 3 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.nij_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nij_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320337430 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320337430;

## Step 2: Create NIJ Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nij_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320337430  -- National Institute of Justice
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.award_number))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.award_number) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.solicitation RLIKE '(?i)fellowship' THEN 'fellowship' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.solicitation), '') as funder_scheme,
    'nij_awards_list' as provenance,
    TRY_TO_DATE(CONCAT(g.fiscal_year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.fiscal_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- org-level award: recipient organisation only (no named PI published).
    -- struct field order MUST match openalex_awards_raw.
    CASE WHEN NULLIF(TRIM(g.recipient), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.recipient),
            'country', CASE WHEN g.state IN ('FM', 'MH', 'PW') THEN g.state
                            WHEN g.state IS NOT NULL THEN 'US' END,
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.award_number))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.nij_raw g
CROSS JOIN src_funder f
WHERE g.excluded = '0'
  AND g.award_number IS NOT NULL AND TRIM(g.award_number) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is an NIJ / OJP award number; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.nij_awards
WHERE NOT funder_award_id RLIKE '^([0-9]{2,4}|15PNIJ|DJO)-'
LIMIT 20;

In [ ]:
%sql
-- Scope check: no formula / Coverdell / backlog-reduction programme slipped through; expect 0 rows.
SELECT funder_award_id, funder_scheme, display_name FROM openalex.awards.nij_awards
WHERE (funder_scheme RLIKE '(?i)coverdell|backlog reduction|formula')
  AND NOT funder_scheme RLIKE '(?i)research|evaluation|development|fellowship'
LIMIT 20;

In [ ]:
%sql
-- Never shrink what may already be published: refuse to replace a larger earlier load.
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.awards.nij_awards) >=
    (SELECT COUNT(*) FROM openalex.awards.openalex_awards_raw WHERE provenance = 'nij_awards_list' AND priority = 537),
    'nij_awards is smaller than the rows already in openalex_awards_raw; refusing to shrink'
) as no_shrink;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nij_awards_list' AND priority = 537;

-- Insert into openalex_awards_raw with priority.
-- Priority 537: direct-from-funder ingest of NIJ's funded-projects list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 537 outranks the
-- crossref/europepmc/datacite acknowledgement shells (priority 0) sharing these ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    537 as priority
FROM openalex.awards.nij_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_nij_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.nij_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year,
       lead_investigator.affiliation.name, landing_page_url
FROM openalex.awards.nij_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.nij_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.nij_awards GROUP BY 1 ORDER BY cnt DESC LIMIT 40;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.nij_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check (org-level: recipient organisation and display_name).
SELECT lead_investigator.affiliation.name AS recipient, COUNT(*) AS n
FROM openalex.awards.nij_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.nij_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 100% title/amount/start_year/recipient).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.affiliation.name) as has_recipient
FROM openalex.awards.nij_awards;

In [ ]:
%sql
-- Citation stubs this ingest collapses onto (same award id).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_nij_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.nij_awards t ON t.id = c.id
WHERE c.funder_id = 4320337430 AND c.priority < 3
GROUP BY c.provenance ORDER BY stub_rows DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nij_awards_list'
GROUP BY provenance, priority;